# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library based on the FAIR² Croissant schema definition.

### Dataset Source
The dataset is defined via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset metadata and record set definitions using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata and instantiate the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s referenced in the Croissant schema.

A dataset may have multiple record sets (`cr:RecordSet`). For accurate and robust downstream analysis, we enumerate each, print its `@id`, `name`, and fields, and show an example record.

In [ ]:
# List all record sets with their @id
print("Available record sets and their field @ids:")

record_sets = []
if hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set
else:
    # Try alternative key commonly used in croissant: `record_sets` or plural
    if hasattr(metadata, 'record_sets'):
        record_sets = metadata.record_sets

if not record_sets:
    raise RuntimeError("No record sets found in this dataset metadata.")

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id if hasattr(rs, 'id') else getattr(rs, '@id', None)}")
    print(f"  name: {getattr(rs, 'name', None)}")
    print(f"  description: {getattr(rs, 'description', None)}")
    fields = getattr(rs, 'fields', getattr(rs, 'field', []))
    if fields:
        print("  Fields & their @ids:")
        for f in fields:
            print(f"    - {f.id if hasattr(f, 'id') else getattr(f, '@id', None)}: {getattr(f, 'name', None)} [{getattr(f, 'data_type', None)}]")
    else:
        print("  No field definitions found.")
    rsid = rs.id if hasattr(rs, 'id') else getattr(rs, '@id', None)
    record_set_ids.append(rsid)
    print("  Sample record:")
    try:
        sample_record = next(dataset.records(record_set=rsid))
        print("   ", json.dumps(sample_record, indent=2)[:800])
    except Exception as e:
        print(f"    <Could not retrieve sample record: {e}>")
    print("\n---------------------\n")

print("All available record set @ids:")
print(record_set_ids)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis.

Here, we extract the first record set as a demonstration, referencing by its `@id`.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set '{record_set_id}' with shape {df.shape}")
    else:
        print(f"Record set '{record_set_id}' is empty or not loadable.")

# Preview the first available DataFrame, using its @id
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No DataFrames created.")

## 4. Exploratory Data Analysis (EDA)
We'll now demonstrate simple data filtering, normalization, and grouping using fields referenced by their `@id`.

You can look up the field `@id` and field name in section 2 above.

In [ ]:
# For demonstration, select an appropriate record set and its numeric field (referenced by @id)
# Example: Choose the first record set and its first numeric field as group
import numpy as np

chosen_record_set_id = None
numeric_field_id = None
group_field_id = None

# Attempt to automatically detect a numeric field
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    df = dataframes[chosen_record_set_id]
    # Try to identify numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert columns containing 'count', 'abundance', or 'weight' to numeric
        for col in df.columns:
            if any(x in col.lower() for x in ['count', 'abundance', 'mw', 'weight', 'coverage', 'pi']):
                df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use the first detected numeric field
        print(f"Selected numeric field (by column/@id): {numeric_field_id}")
    # Assign a group field (for example, any column containing 'group' or 'sample' or 'type')
    for col in df.columns:
        if any(x in col.lower() for x in ['group', 'sample', 'type']):
            group_field_id = col
            print(f"Selected group field (by column/@id): {group_field_id}")
            break

    # Demonstration threshold
    threshold = df[numeric_field_id].mean() if numeric_field_id and not df[numeric_field_id].isnull().all() else 0

    # Filter
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())
        # If group column exists, group by it
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No numeric field or record set available for analysis.")

## 5. Visualization
Visualize distributions and relationships between fields in the dataset.

Below is a simple histogram and scatterplot using the selected fields. Adapt as needed for your data.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and numeric_field_id:
    df = dataframes[chosen_record_set_id]
    # Hist plot
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there's another numeric field for scatter
    additional_numeric_fields = [col for col in df.select_dtypes(include=[np.number]).columns if col != numeric_field_id]
    if additional_numeric_fields:
        plt.figure(figsize=(6,4))
        plt.scatter(df[numeric_field_id], df[additional_numeric_fields[0]])
        plt.xlabel(numeric_field_id)
        plt.ylabel(additional_numeric_fields[0])
        plt.title(f"Scatter of {numeric_field_id} vs {additional_numeric_fields[0]}")
        plt.show()
else:
    print("Visualization skipped: No numeric field available.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the FAIR² Croissant dataset on Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells using the `mlcroissant` library.

- All data entity references (record sets, fields, columns) were made by their `@id` attributes to ensure clarity and reproducibility.
- The dataset exposes protein abundance and related metadata across samples, suitable for analysis and visualization.

You can use the approach demonstrated here as a template for interacting with any Croissant-compliant scientific dataset!